<a href="https://colab.research.google.com/github/JssSalas/bash/blob/main/Reto_An%C3%A1lisis_de_Premios_Novel_con_Mongo_DB_y_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Jesús Carlos Salas García

Reto: Análisis de Premios Novel con Mongo DB y Python

In [1]:
# Reto: Análisis de Premios Nobel con MongoDB y Python

Parte 1 Configuración del Entorno en Google Colab

In [2]:
# 1: Instalación de MongoDB
!apt update
!apt install wget curl gnupg2 software-properties-common apt-transport-https ca-certificates lsb-release -y
!curl -fsSL https://www.mongodb.org/static/pgp/server-6.0.asc | sudo gpg --dearmor -o /etc/apt/trusted.gpg.d/mongodb-6.gpg
!echo "deb [ arch=amd64,arm64 ] https://repo.mongodb.org/apt/ubuntu $(lsb_release -cs)/mongodb-org/6.0 multiverse" | tee /etc/apt/sources.list.d/mongodb-org-6.0.list
!apt update
!apt install mongodb-org -y
!mkdir -p /data
!mkdir -p /data/db
!mongod --fork --logpath /var/log/mongodb/mongod.log

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,529 kB]
Get:7 https://cli.github.com/packages stable InRelease [3,917 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,734 kB]
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backp

In [3]:
# 2: Montar Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

print("Drive montado")

Mounted at /content/gdrive
Drive montado


In [4]:
# 3: Importar datos con mongoimport
!mongoimport --db nobel --collection laureates --type=json --file=/content/gdrive/MyDrive/Data/json_laureates.json --jsonArray
!mongoimport --db nobel --collection awards --type=json --file=/content/gdrive/MyDrive/Data/json_award.json --jsonArray

2026-02-19T01:28:34.184+0000	connected to: mongodb://localhost/
2026-02-19T01:28:34.848+0000	943 document(s) imported successfully. 0 document(s) failed to import.
2026-02-19T01:28:34.909+0000	connected to: mongodb://localhost/
2026-02-19T01:28:35.297+0000	646 document(s) imported successfully. 0 document(s) failed to import.


In [5]:
# Instalar dependencias
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 16.3 MB/s eta 0:00:00


Parte 2: Conexión a MongoDB desde Python

In [6]:
# 4: Importar librerías y establecer conexión

from pymongo import MongoClient
import pandas as pd
from datetime import datetime

# Conexión a MongoDB local
client = MongoClient('localhost', 27017)

# Referencia a la base de datos nobel
db_nobel = client.nobel

# Referencias a las colecciones
col_laureates = db_nobel.laureates
col_awards = db_nobel.awards

print("✓ Conexión establecida exitosamente")
print(f"✓ Colección laureates: {col_laureates.count_documents({})} documentos")
print(f"✓ Colección awards: {col_awards.count_documents({})} documentos")


✓ Conexión establecida exitosamente
✓ Colección laureates: 943 documentos
✓ Colección awards: 646 documentos


Parte 3: Análisis de datos

In [7]:
# 5: Consulta de mujeres ganadoras
query_mujeres = {"gender": "female"}
projection_mujeres = {
    "fullName.en": 1,
    "gender": 1,
    "birth.place.country.en": 1,
    "nobelPrizes.category.en": 1,
    "nobelPrizes.awardYear": 1,
    "_id": 0
}

lista_mujeres_ganadoras = list(col_laureates.find(query_mujeres, projection_mujeres))

print("=" * 80)
print("MUJERES GANADORAS DEL PREMIO NOBEL")
print("=" * 80)
for i, mujer in enumerate(lista_mujeres_ganadoras[:10], 1):
    nombre = mujer.get('fullName', {}).get('en', 'N/A')
    pais = mujer.get('birth', {}).get('place', {}).get('country', {}).get('en', 'N/A')
    print(f"{i}. {nombre} - País: {pais}")
    if 'nobelPrizes' in mujer:
        for prize in mujer['nobelPrizes']:
            categoria = prize.get('category', {}).get('en', 'N/A')
            anio = prize.get('awardYear', 'N/A')
            print(f"   → {categoria} ({anio})")

print(f"\nTotal de mujeres ganadoras: {len(lista_mujeres_ganadoras)}")


MUJERES GANADORAS DEL PREMIO NOBEL
1. Ada E. Yonath - País: British Mandate of Palestine
   → Chemistry (2009)
2. Alice Munro - País: Canada
   → Literature (2013)
3. Alva Myrdal - País: Sweden
   → Peace (1982)
4. Aung San Suu Kyi  - País: Burma
   → Peace (1991)
5. Barbara McClintock - País: USA
   → Physiology or Medicine (1983)
6. Baroness Bertha Sophie Felicita von Suttner, née Countess Kinsky von Chinic und Tettau - País: Austrian Empire
   → Peace (1905)
7. Elizabeth Williams - País: Northern Ireland
   → Peace (1976)
8. Carol W. Greider - País: USA
   → Physiology or Medicine (2009)
9. Christiane Nüsslein-Volhard - País: Germany
   → Physiology or Medicine (1995)
10. Donna Strickland - País: Canada
   → Physics (2018)

Total de mujeres ganadoras: 53


In [8]:
# 6: Calcular porcentaje de mujeres
total_laureados = col_laureates.count_documents({})
total_mujeres = len(lista_mujeres_ganadoras)
porcentaje_mujeres = (total_mujeres / total_laureados) * 100 if total_laureados > 0 else 0

print("=" * 80)
print("REPRESENTACIÓN DE MUJERES EN PREMIOS NOBEL")
print("=" * 80)
print(f"Total de laureados: {total_laureados}")
print(f"Mujeres ganadoras: {total_mujeres}")
print(f"Porcentaje de mujeres: {porcentaje_mujeres:.2f}%")
print(f"Porcentaje de hombres: {100 - porcentaje_mujeres:.2f}%")
print("\n⚠️ HALLAZGO: Solo {:.2f}% son mujeres - Subrepresentación significativa".format(porcentaje_mujeres))


REPRESENTACIÓN DE MUJERES EN PREMIOS NOBEL
Total de laureados: 943
Mujeres ganadoras: 53
Porcentaje de mujeres: 5.62%
Porcentaje de hombres: 94.38%

⚠️ HALLAZGO: Solo 5.62% son mujeres - Subrepresentación significativa


In [9]:
# 7: Agregación para países con menos premios
pipeline_paises_menos_premios = [
    {"$unwind": "$nobelPrizes"},
    {"$project": {"pais": "$birth.place.country.en"}},
    {"$match": {"pais": {"$exists": True, "$ne": None}}},
    {"$group": {"_id": "$pais", "cantidad_premios": {"$sum": 1}}},
    {"$sort": {"cantidad_premios": 1}},
    {"$limit": 10},
    {"$project": {"_id": 0, "pais": "$_id", "cantidad_premios": 1}}
]

top10_paises_menos_premios = list(col_laureates.aggregate(pipeline_paises_menos_premios))

print("=" * 80)
print("TOP 10 PAÍSES CON MENOS PREMIOS NOBEL")
print("=" * 80)
for i, pais_info in enumerate(top10_paises_menos_premios, 1):
    print(f"{i}. {pais_info['pais']}: {pais_info['cantidad_premios']} premio(s)")

print("\n⚠️ Estos países tienen representación mínima")


TOP 10 PAÍSES CON MENOS PREMIOS NOBEL
1. Iraq: 1 premio(s)
2. Burma: 1 premio(s)
3. Gold Coast: 1 premio(s)
4. Bulgaria: 1 premio(s)
5. Nigeria: 1 premio(s)
6. British West Indies: 1 premio(s)
7. Morocco: 1 premio(s)
8. Ethiopia: 1 premio(s)
9. Cyprus: 1 premio(s)
10. Crete: 1 premio(s)

⚠️ Estos países tienen representación mínima


In [10]:
# 8: Calcular porcentaje de países con menos premios
pipeline_total_paises = [
    {"$unwind": "$nobelPrizes"},
    {"$project": {"pais": "$birth.place.country.en"}},
    {"$match": {"pais": {"$exists": True, "$ne": None}}},
    {"$group": {"_id": "$pais"}},
    {"$count": "total"}
]

resultado_total_paises = list(col_laureates.aggregate(pipeline_total_paises))
total_paises_unicos = resultado_total_paises[0]['total'] if resultado_total_paises else 0

cantidad_paises_top10_menos = len(top10_paises_menos_premios)
porcentaje_paises_menos_representados = (
    (cantidad_paises_top10_menos / total_paises_unicos) * 100
    if total_paises_unicos > 0 else 0
)

print("=" * 80)
print("ANÁLISIS DE REPRESENTACIÓN POR PAÍSES")
print("=" * 80)
print(f"Total de países con premios Nobel: {total_paises_unicos}")
print(f"Países en top 10 con menos premios: {cantidad_paises_top10_menos}")
print(f"Porcentaje que representan: {porcentaje_paises_menos_representados:.2f}%")
print("\n⚠️ HALLAZGO: Estos países tienen baja representación histórica")


ANÁLISIS DE REPRESENTACIÓN POR PAÍSES
Total de países con premios Nobel: 95
Países en top 10 con menos premios: 10
Porcentaje que representan: 10.53%

⚠️ HALLAZGO: Estos países tienen baja representación histórica


In [11]:
# 9: Agregación de mujeres por categoría
pipeline_mujeres_por_categoria = [
    {"$unwind": "$nobelPrizes"},
    {"$group": {
        "_id": {"categoria": "$nobelPrizes.category.en", "genero": "$gender"},
        "cantidad": {"$sum": 1}
    }},
    {"$group": {
        "_id": "$_id.categoria",
        "premios": {"$push": {"genero": "$_id.genero", "cantidad": "$cantidad"}},
        "total": {"$sum": "$cantidad"}
    }},
    {"$sort": {"_id": 1}}
]

resultado_genero_categoria = list(col_laureates.aggregate(pipeline_mujeres_por_categoria))

categorias_analisis = []
for cat in resultado_genero_categoria:
    categoria_nombre = cat['_id']
    total_categoria = cat['total']
    mujeres_count = 0
    hombres_count = 0
    for premio in cat['premios']:
        genero = premio.get('genero') # Safely get 'genero', returns None if not present
        if genero == 'female':
            mujeres_count = premio['cantidad']
        elif genero == 'male':
            hombres_count = premio['cantidad']
    porcentaje_mujeres_cat = (mujeres_count / total_categoria * 100) if total_categoria > 0 else 0
    categorias_analisis.append({
        'categoria': categoria_nombre,
        'total': total_categoria,
        'mujeres': mujeres_count,
        'hombres': hombres_count,
        'porcentaje_mujeres': porcentaje_mujeres_cat
    })

categorias_menor_representacion_mujeres = sorted(
    categorias_analisis, key=lambda x: x['porcentaje_mujeres']
)

print("=" * 80)
print("CATEGORÍAS CON MENOR REPRESENTACIÓN DE MUJERES")
print("=" * 80)
for i, cat in enumerate(categorias_menor_representacion_mujeres, 1):
    print(f"{i}. {cat['categoria']}")
    print(f"   Total premios: {cat['total']}")
    print(f"   Mujeres: {cat['mujeres']} ({cat['porcentaje_mujeres']:.2f}%)")
    print(f"   Hombres: {cat['hombres']} ({100 - cat['porcentaje_mujeres']:.2f}%)")
    print()

print("⚠️ HALLAZGO: Física y Economía tienen la menor representación femenina")

CATEGORÍAS CON MENOR REPRESENTACIÓN DE MUJERES
1. Physics
   Total premios: 213
   Mujeres: 3 (1.41%)
   Hombres: 210 (98.59%)

2. Economic Sciences
   Total premios: 84
   Mujeres: 2 (2.38%)
   Hombres: 82 (97.62%)

3. Chemistry
   Total premios: 184
   Mujeres: 5 (2.72%)
   Hombres: 179 (97.28%)

4. Physiology or Medicine
   Total premios: 219
   Mujeres: 12 (5.48%)
   Hombres: 207 (94.52%)

5. Peace
   Total premios: 134
   Mujeres: 17 (12.69%)
   Hombres: 90 (87.31%)

6. Literature
   Total premios: 116
   Mujeres: 15 (12.93%)
   Hombres: 101 (87.07%)

⚠️ HALLAZGO: Física y Economía tienen la menor representación femenina


In [12]:
# 10: Países con menor representación de mujeres (solo países con ≥5 premios)
pipeline_genero_por_pais = [
    {"$unwind": "$nobelPrizes"},
    {"$project": {
        "pais": "$birth.place.country.en",
        "genero": "$gender"
    }},
    {"$match": {"pais": {"$exists": True, "$ne": None}}},
    {"$group": {
        "_id": {"pais": "$pais", "genero": "$genero"},
        "cantidad": {"$sum": 1}
    }},
    {"$group": {
        "_id": "$_id.pais",
        "distribuciones": {"$push": {"genero": "$_id.genero", "cantidad": "$cantidad"}},
        "total": {"$sum": "$cantidad"}
    }},
    {"$match": {"total": {"$gte": 5}}},
    {"$sort": {"total": -1}}
]

resultado_genero_pais = list(col_laureates.aggregate(pipeline_genero_por_pais))

paises_analisis_genero = []
for pais_data in resultado_genero_pais:
    pais_nombre = pais_data['_id']
    total_pais = pais_data['total']
    mujeres_count = 0
    hombres_count = 0
    for dist in pais_data['distribuciones']:
        if dist['genero'] == 'female':
            mujeres_count = dist['cantidad']
        elif dist['genero'] == 'male':
            hombres_count = dist['cantidad']
    porcentaje_mujeres_pais = (mujeres_count / total_pais * 100) if total_pais > 0 else 0
    paises_analisis_genero.append({
        'pais': pais_nombre,
        'total': total_pais,
        'mujeres': mujeres_count,
        'hombres': hombres_count,
        'porcentaje_mujeres': porcentaje_mujeres_pais
    })

paises_menor_representacion_mujeres = sorted(
    paises_analisis_genero, key=lambda x: x['porcentaje_mujeres']
)[:10]

print("=" * 80)
print("TOP 10 PAÍSES CON MENOR REPRESENTACIÓN DE MUJERES")
print("(Países con al menos 5 premios Nobel)")
print("=" * 80)
for i, pais in enumerate(paises_menor_representacion_mujeres, 1):
    print(f"{i}. {pais['pais']}")
    print(f"   Total premios: {pais['total']}")
    print(f"   Mujeres: {pais['mujeres']} ({pais['porcentaje_mujeres']:.2f}%)")
    print(f"   Hombres: {pais['hombres']} ({100 - pais['porcentaje_mujeres']:.2f}%)")
    print()

print("⚠️ HALLAZGO: Países desarrollados también muestran baja representación femenina")


TOP 10 PAÍSES CON MENOR REPRESENTACIÓN DE MUJERES
(Países con al menos 5 premios Nobel)
1. United Kingdom
   Total premios: 87
   Mujeres: 0 (0.00%)
   Hombres: 87 (100.00%)

2. Japan
   Total premios: 27
   Mujeres: 0 (0.00%)
   Hombres: 27 (100.00%)

3. Switzerland
   Total premios: 19
   Mujeres: 0 (0.00%)
   Hombres: 19 (100.00%)

4. the Netherlands
   Total premios: 18
   Mujeres: 0 (0.00%)
   Hombres: 18 (100.00%)

5. Russia
   Total premios: 17
   Mujeres: 0 (0.00%)
   Hombres: 17 (100.00%)

6. Prussia
   Total premios: 13
   Mujeres: 0 (0.00%)
   Hombres: 13 (100.00%)

7. Scotland
   Total premios: 11
   Mujeres: 0 (0.00%)
   Hombres: 11 (100.00%)

8. India
   Total premios: 10
   Mujeres: 0 (0.00%)
   Hombres: 10 (100.00%)

9. Belgium
   Total premios: 9
   Mujeres: 0 (0.00%)
   Hombres: 9 (100.00%)

10. Spain
   Total premios: 7
   Mujeres: 0 (0.00%)
   Hombres: 7 (100.00%)

⚠️ HALLAZGO: Países desarrollados también muestran baja representación femenina


In [13]:
# 11: Años donde el número de países representados es menor al promedio histórico
pipeline_paises_por_anio = [
    {"$unwind": "$nobelPrizes"},
    {"$project": {
        "anio": "$nobelPrizes.awardYear",
        "pais": "$birth.place.country.en"
    }},
    {"$match": {
        "pais": {"$exists": True, "$ne": None},
        "anio": {"$exists": True, "$ne": None}
    }},
    {"$group": {
        "_id": {"anio": "$anio", "pais": "$pais"}
    }},
    {"$group": {
        "_id": "$_id.anio",
        "paises_representados": {"$sum": 1}
    }},
    {"$sort": {"_id": 1}},
    {"$project": {
        "_id": 0,
        "anio": "$_id",
        "paises_representados": 1
    }}
]

distribucion_paises_anio = list(col_laureates.aggregate(pipeline_paises_por_anio))

total_paises_todos_anios = sum(d['paises_representados'] for d in distribucion_paises_anio)
promedio_paises_por_anio = (
    total_paises_todos_anios / len(distribucion_paises_anio)
    if distribucion_paises_anio else 0
)

anios_menor_diversidad = [
    d for d in distribucion_paises_anio
    if d['paises_representados'] < promedio_paises_por_anio
]

print("=" * 80)
print("ANÁLISIS DE DIVERSIDAD GEOGRÁFICA POR AÑO")
print("=" * 80)
print(f"Promedio histórico de países representados por año: {promedio_paises_por_anio:.2f}")
print(f"\nAños con representación MENOR al promedio: {len(anios_menor_diversidad)}")
print("\nPrimeros 20 años con menor diversidad:")
print("-" * 80)

for i, anio_data in enumerate(
    sorted(anios_menor_diversidad, key=lambda x: x['paises_representados'])[:20],
    1
):
    diff = anio_data['paises_representados'] - promedio_paises_por_anio
    print(f"{i}. Año {anio_data['anio']}: {anio_data['paises_representados']} países "
          f"(Diferencia: {diff:.2f})")

print(f"\n⚠️ HALLAZGO: {len(anios_menor_diversidad)} años tuvieron baja diversidad geográfica")


ANÁLISIS DE DIVERSIDAD GEOGRÁFICA POR AÑO
Promedio histórico de países representados por año: 5.41

Años con representación MENOR al promedio: 64

Primeros 20 años con menor diversidad:
--------------------------------------------------------------------------------
1. Año 1916: 1 países (Diferencia: -4.41)
2. Año 1917: 2 países (Diferencia: -3.41)
3. Año 1918: 2 países (Diferencia: -3.41)
4. Año 1946: 2 países (Diferencia: -3.41)
5. Año 1954: 2 países (Diferencia: -3.41)
6. Año 1914: 3 países (Diferencia: -2.41)
7. Año 1924: 3 países (Diferencia: -2.41)
8. Año 1931: 3 países (Diferencia: -2.41)
9. Año 1932: 3 países (Diferencia: -2.41)
10. Año 1934: 3 países (Diferencia: -2.41)
11. Año 1972: 3 países (Diferencia: -2.41)
12. Año 1976: 3 países (Diferencia: -2.41)
13. Año 1993: 3 países (Diferencia: -2.41)
14. Año 2006: 3 países (Diferencia: -2.41)
15. Año 1901: 4 países (Diferencia: -1.41)
16. Año 1910: 4 países (Diferencia: -1.41)
17. Año 1912: 4 países (Diferencia: -1.41)
18. Año 191

In [14]:
# 12: Resumen de todas las variables importantes
print("=" * 80)
print("RESUMEN DE VARIABLES ALMACENADAS")
print("=" * 80)

variables_resumen = {
    "lista_mujeres_ganadoras": f"Lista con {len(lista_mujeres_ganadoras)} mujeres ganadoras",
    "total_laureados": f"Total de laureados: {total_laureados}",
    "total_mujeres": f"Total de mujeres: {total_mujeres}",
    "porcentaje_mujeres": f"Porcentaje de mujeres: {porcentaje_mujeres:.2f}%",
    "top10_paises_menos_premios": f"Lista con 10 países con menos premios",
    "total_paises_unicos": f"Total de países con premios: {total_paises_unicos}",
    "porcentaje_paises_menos_representados": f"{porcentaje_paises_menos_representados:.2f}%",
    "categorias_menor_representacion_mujeres": f"Lista de {len(categorias_menor_representacion_mujeres)} categorías ordenadas",
    "paises_menor_representacion_mujeres": f"Lista de 10 países con menos mujeres",
    "promedio_paises_por_anio": f"Promedio: {promedio_paises_por_anio:.2f} países/año",
    "anios_menor_diversidad": f"Lista con {len(anios_menor_diversidad)} años con baja diversidad"
}

for variable, descripcion in variables_resumen.items():
    print(f"✓ {variable}: {descripcion}")


RESUMEN DE VARIABLES ALMACENADAS
✓ lista_mujeres_ganadoras: Lista con 53 mujeres ganadoras
✓ total_laureados: Total de laureados: 943
✓ total_mujeres: Total de mujeres: 53
✓ porcentaje_mujeres: Porcentaje de mujeres: 5.62%
✓ top10_paises_menos_premios: Lista con 10 países con menos premios
✓ total_paises_unicos: Total de países con premios: 95
✓ porcentaje_paises_menos_representados: 10.53%
✓ categorias_menor_representacion_mujeres: Lista de 6 categorías ordenadas
✓ paises_menor_representacion_mujeres: Lista de 10 países con menos mujeres
✓ promedio_paises_por_anio: Promedio: 5.41 países/año
✓ anios_menor_diversidad: Lista con 64 años con baja diversidad
